In [ ]:
import numpy as np
import pandas as pd
import logging

from simpletransformers.classification import MultiLabelClassificationModel, MultiLabelClassificationArgs

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_csv('train.En.csv')
df

In [ ]:
df = df.dropna(subset = ['tweet', 'sarcasm'])[['tweet', 'sarcasm', 'irony', 'satire', 'understatement', 'overstatement', 'rhetorical_question']]
df.head(10)

In [ ]:
data_categ = df[['sarcasm', 'irony', 'satire', 'understatement', 'overstatement', 'rhetorical_question']].values.tolist()
data_categ

In [ ]:
data_tw = df['tweet'].values.tolist()
data_tw

In [ ]:
data = pd.DataFrame()
data['text'] = data_tw
data['labels'] = data_categ
data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data.text, data.labels, test_size=0.25, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
data_train = pd.DataFrame()
data_train['text'] = list(X_train)
data_train['labels'] = list(y_train)

data_val = pd.DataFrame()
data_val['text'] = list(X_val)
data_val['labels'] = list(y_val)

data_test = pd.DataFrame()
data_test['text'] = list(X_test)
data_test['labels'] = list(y_test)

In [ ]:
logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

model_args = {'overwrite_output_dir': True, 'num_train_epochs':5}

# CPU run
model = MultiLabelClassificationModel("roberta", "roberta-base", use_cuda=False, num_labels=6, args=model_args)

# GPU run
# model = MultiLabelClassificationModel("roberta", "roberta-base", num_labels=6, args=model_args)

model.train_model(data_train)

result, model_outputs, wrong_predictions = model.eval_model(data_val)

In [ ]:
predictions, raw_outputs = model.predict(data_test.text.to_list())

In [ ]:
y_true = [list(map(int, sublist)) for sublist in data_test.labels.to_list()]

In [ ]:
f1_score(y_true, predictions, average='micro')